# 배경 생성 PoC — sd-turbo
그라데이션(임시 배경)을 AI 생성 배경으로 교체하기 위한 실험.
프롬프트 → 배경 생성 → 누끼·문구와 합성까지 확인한다.

In [ ]:
import torch
from diffusers import AutoPipelineForText2Image

pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sd-turbo", torch_dtype=torch.float16
).to("cuda")

In [ ]:
prompt = "warm cozy cafe interior, wooden table, soft morning light, blurred background, product photo backdrop"
bg = pipe(prompt=prompt, num_inference_steps=2, guidance_scale=0.0).images[0]
bg

In [ ]:
from PIL import Image
from app_core.background import remove_background
from app_core.compose import compose_ad

cut = remove_background(Image.open("테스트사진.jpg"))
ad = compose_ad(cut, "크로플 출시 기념!", "지금 바로 3,500원에 만나요!", background=bg)
ad.save("첫_AI배경_광고.png")

small = ad.copy()
small.thumbnail((420, 420))
small

## 실험 기록 (2026-08-10)
- sd-turbo 2스텝, 영어 프롬프트 → 카페 배경 성공 (한글 프롬프트는 엉뚱한 그림 — 번역 부품 필요)
- AI 배경 + 누끼 + 두 줄 문구 첫 합성 성공 (첫_AI배경_광고.png)
- 개선 2호 후보: 사진 배경에서 글자 가독성 (반투명 띠 / 흰 글자+그림자)
- 개선 3호 후보: 제품 아래 그림자 (스티커 느낌 제거)